# L11b Lab: Implementing a Markov Word Generator
In this lab, we construct a program that generates random character sequences of defined length using a discrete Markov chain.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Build a transition matrix from data:__ A character-level Markov chain needs the probability of each next character given the current one. We'll estimate these probabilities from about 370,000 English words.
> * __Generate words with a Markov chain:__ A Markov chain generates a sequence by drawing each next state from the row of the transition matrix for the current state. We'll start every word with the same letter and draw the remaining characters one at a time.
> * __Compare generated words with real words:__ Generated words follow the letter patterns in the data, but they are not guaranteed to be words. We'll count how many of them are in the vocabulary and how that count changes with word length.

Previously, we introduced Markov chains and their transition matrices. Here, the states of our chain are the letters `a` to `z`.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [1]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

Your transition matrix function goes in [src/Compute.jl](src/Compute.jl). We call [the `L11bMarkov.my_transition_matrix(...)` function](src/Compute.jl) by its qualified name, so that re-running the setup cell picks up your edits.

### Data
We included a JSON file containing approximately 370K English words [originally downloaded from here](https://github.com/dwyl/english-words) in [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). We'll use this data to train our Markov word generator.

[The `MyEnglishLanguageVocabularyModel(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyEnglishLanguageVocabularyModel) reads the words from the JSON file and groups them by their first character. We'll save the vocabulary data in the `vocabulary_data_dictionary::Dict{Char, Set{String}}` variable:

In [2]:
vocabulary_data_dictionary = MyEnglishLanguageVocabularyModel();

What's in the `vocabulary_data_dictionary` variable? Let's look at words that start with the character `a`:

In [3]:
vocabulary_data_dictionary['a'] # Wow! We have > 25000 words that start with the character 'a'!

Set{String} with 25416 elements:
  "archetypes"
  "acrodont"
  "apogaic"
  "acousma"
  "acutangular"
  "alteza"
  "anhidrosis"
  "abracadabra"
  "agrostographies"
  "anthropologically"
  "assentation"
  "attache"
  "acanthad"
  "aristotelism"
  "alterity"
  "attargul"
  "arditi"
  "abducted"
  "amotus"
  ⋮ 

___

## Task 1: Compute the Character Transition Matrix
In this task, we'll compute the transition matrix $\mathbf{P}$ for our Markov chain. The transition matrix defines the probabilities of moving from one character to another in our generated words.

Let's compute the transition probabilities using the first two characters of each word to make our lives easier. For letters $i$ and $j$, let $n_{ij}$ be the number of words that start with $i$ and have $j$ as their second character. The probability of moving from $i$ to $j$ is the fraction of the words starting with $i$ whose second character is $j$:
$$
\begin{equation*}
p_{ij} = \frac{n_{ij}}{\sum_{k} n_{ik}}
\end{equation*}
$$
where the sum runs over all 26 letters, so every row of $\mathbf{P}$ sums to one. We skip one-letter words.

__Your turn:__ Complete TODO 1 through TODO 3 in [the `my_transition_matrix(...)` function](src/Compute.jl): take the second character of each word, add one to its count, and divide each row of counts by its total. Then re-run the setup cell and the cell below, which saves the result in the `P::Array{Float64, 2}` variable:

In [4]:
P, characters = let

    # build the character list and transition matrix -
    characters = [Char(i) for i in 'a':'z']; # list of characters to consider
    P = L11bMarkov.my_transition_matrix(vocabulary_data_dictionary, characters); # the untouched stub throws an error

    P, characters # return the transition matrix, and the character list
end;

ErrorException: Oooops! The `my_transition_matrix(...)` function is not implemented yet - we'd better fix that. Complete TODO 1 through TODO 3.

### Check: Your Transition Matrix
[The `vocabulary_transition_matrix(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/text/#VLDataScienceMachineLearningPackage.vocabulary_transition_matrix) in the course package computes the same matrix, so our $\mathbf{P}$ should match it entry by entry, and every row should sum to one. We also try a small test vocabulary that we can count by hand:

In [5]:
@testset "My transition matrix" begin

    # our matrix matches the course package, and every row is a probability distribution -
    @test P ≈ vocabulary_transition_matrix(vocabulary_data_dictionary, characters)
    @test all(P .≥ 0) && all(abs.(sum(P, dims = 2) .- 1) .< 1e-10)

    # a test vocabulary: "a" is too short and the '-' in "c-a" is not a state, so both are skipped -
    test_vocabulary = Dict('a' => Set(["a", "ab", "abc", "ac"]), 'b' => Set(["ba", "bb"]), 'c' => Set(["ca", "c-a"]));
    P_test = L11bMarkov.my_transition_matrix(test_vocabulary, ['a', 'b', 'c']);
    @test P_test ≈ [0 2/3 1/3; 1/2 1/2 0; 1 0 0]
end;

My transition matrix: Error During Test at /Users/jeffreyvarner/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-11/L11b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X14sZmlsZQ==.jl:4
  Test threw exception
  Expression: P ≈ vocabulary_transition_matrix(vocabulary_data_dictionary, characters)
  UndefVarError: `P` not defined in `Main`
  Suggestion: check for spelling errors or missing imports.
  Stacktrace:
   [1] top-level scope
     @ ~/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-11/L11b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X14sZmlsZQ==.jl:4
   [2] macro expansion
     @ ~/.julia/juliaup/julia-1.12.7+0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/stdlib/v1.12/Test/src/Test.jl:1777 [inlined]
   [3] macro expansion
     @ ~/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-11/L11b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X14sZmlsZQ==.jl:4 [inlined]
   [4] macro expansio

Test.TestSetException: Some tests did not pass: 0 passed, 0 failed, 3 errored, 0 broken.

The transition matrix $\mathbf{P}$ is a 26×26 matrix, where the rows and columns correspond to the characters `a` to `z`. Let's examine a few transition probabilities from the matrix $\mathbf{P}$:

In [6]:
let

    # initialize -
    start_char = 'b'; # starting character (change this to see other characters)
    next_char = 'r'; # next character (change this to see other characters)
    i = findfirst(x -> x == start_char, characters); # index of the start_char
    j = findfirst(x -> x == next_char, characters); # index of the next_char

    # print the transition probability -
    println("Transition probability P('$start_char' -> '$next_char') = $(P[i, j] |> x-> round(x, digits=4))");
end

UndefVarError: UndefVarError: `characters` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

What's the most likely next character after each character? We find the largest entry in each row of $\mathbf{P}$ with [the `argmax(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.argmax):

In [7]:
let
    for i ∈ eachindex(characters)
        char = characters[i]
        row = P[i, :]
        max_index = argmax(row);
        max_prob = row[max_index];
        most_likely_char = characters[max_index]
        println("From '$char' the most likely next character is '$most_likely_char' with probability $(round(max_prob, digits=4))")
    end
end

UndefVarError: UndefVarError: `characters` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

__What do we see?__ After `q`, the next character is almost always `u` (probability 0.971). A few other rows are dominated by one character: `u` is followed by `n` with probability 0.885, `n` by `o` with 0.647, and `r` by `e` with 0.614.

### Things to think about

__Question:__ Many English words begin with `un`, `non`, or `re`. How does the way we built $\mathbf{P}$ explain the large probabilities after `u`, `n`, and `r`? Would they be as large if we counted every pair of neighboring characters in every word? Briefly explain.

___

## Task 2: Generate Random Words One Character at a Time
In this task, we'll generate random words using the transition matrix we computed in Task 1. We'll start with an initial character and then use the transition probabilities to select subsequent characters until we reach the desired word length.

> __Helper:__ We've implemented [the `sample_words(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/text/#VLDataScienceMachineLearningPackage.sample_words) to generate random character sequences by iteratively sampling from the transition matrix. The function takes the transition matrix `P::Array{Float64,2}`, a list of characters `characters::Vector{Char}`, and generates `number_of_samples::Int` words of length `length_of_sample_word::Int` starting from `startchar::Char`.

We first set the seed with [the `Random.seed!(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.seed!), so the cell gives the same words every time we run it. Let's generate `number_of_samples` random words with `length_of_sample_word` characters, each starting with `sample_word_start_char`, and store them in `generated_words_set::Set{String}`, which keeps one copy of each word:

In [8]:
# the settings for our words (you can change these) -
seed = 5800; # the random seed: the same seed gives the same words
number_of_samples = 1000; # how many words to generate
length_of_sample_word = 6; # how many characters in each word
sample_word_start_char = 'b'; # every word starts with this character

generated_words_set = let

    # generate samples -
    Random.seed!(seed);
    S = sample_words(P, characters, number_of_samples = number_of_samples,
        length_of_sample_word = length_of_sample_word, startchar = sample_word_start_char);

    # store the generated words in a Set to avoid duplicates -
    generated_words_set = Set{String}();
    for (i, word) ∈ S
        push!(generated_words_set, word);
    end

    generated_words_set; # return the generated words
end;
println("We generated $(length(generated_words_set)) different words. The first ten in alphabetical order:")
first(sort(collect(generated_words_set)), 10)

UndefVarError: UndefVarError: `P` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

__What do we see?__ Wow! That's kind of fun. The printed count shows how many of the generated words are different from each other, and some of them look like English words.

### Things to think about

__Question:__ Each new character in a generated word depends only on the character before it. Which property of Markov chains is this? What does our generator ignore about the word so far? Briefly explain.

___

## Task 3: How Many Generated Words Are Real?
But, I'm curious: how many of the generated words are actually real English words? Let's check the generated words against our vocabulary data and store the real ones in `true_generated_words::Set{String}`.

__Prediction:__ What fraction of our different generated words will be real English words: {less than 5 percent, about a quarter, or more than half}?

In [9]:
true_generated_words = let

    # initialize -
    my_word_set = vocabulary_data_dictionary[sample_word_start_char]; # real words that start with our character
    number_of_words = length(generated_words_set); # number of different generated words
    true_generated_words = Set{String}();

    for word ∈ generated_words_set
        if word ∈ my_word_set
            push!(true_generated_words, word);
        end
    end

    N₊ = length(true_generated_words); # number of true words, that are in the vocabulary model
    println("Real words: $(N₊) of $(number_of_words), or $(round((N₊/number_of_words)*100, digits=2))%");
    true_generated_words; # return the true words
end;

UndefVarError: UndefVarError: `generated_words_set` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

Let's look at some of the __actual__ words, in alphabetical order:

In [10]:
let

    # initialize -
    number_of_words_to_look_at = min(10, length(true_generated_words)); # how many do we want to see?
    real_words = sort(collect(true_generated_words)); # the real words, in alphabetical order

    # populate the data frame -
    df = DataFrame(index = 1:number_of_words_to_look_at, true_word = real_words[1:number_of_words_to_look_at]);

    # make a table -
    pretty_table(
         df;
         backend = :text,
         table_format = TextTableFormat(borders = text_table_borders__compact)
    );
end

UndefVarError: UndefVarError: `true_generated_words` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

__What do we see?__ With our seed, only a small fraction of the different words are real; the printed percentage gives its value. The fraction depends on many factors, including the length of the requested words and the initial character.

__Prediction:__ Suppose we generate shorter words. Will the fraction of real words {increase, decrease, or stay about the same}?

In [11]:
let

    # initialize -
    df = DataFrame();
    my_word_set = vocabulary_data_dictionary[sample_word_start_char]; # real words that start with our character

    # generate words of each length with the same seed, and count the real ones -
    for word_length ∈ 4:8
        Random.seed!(seed);
        S = sample_words(P, characters, number_of_samples = number_of_samples,
            length_of_sample_word = word_length, startchar = sample_word_start_char);
        different_words = Set(values(S)); # one copy of each generated word
        N₊ = count(word -> word ∈ my_word_set, different_words); # the real ones
        push!(df, (word_length = word_length, different_words = length(different_words), real_words = N₊,
            percent_real = round(100*N₊/length(different_words), digits = 2)));
    end

    # make a table -
    pretty_table(
         df;
         backend = :text,
         table_format = TextTableFormat(borders = text_table_borders__compact)
    );
end

UndefVarError: UndefVarError: `P` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

__What do we see?__ Shorter words are real far more often: in the table, the percentage of real words falls quickly as the word length grows.

### Things to think about

__Question:__ A real English word that is missing from our vocabulary counts as not real. Does that make our fraction too high or too low? How could we use more of each word to choose the next character, for example the previous two characters? Briefly explain.

___

## Summary
In this lab, we estimated a character transition matrix from English words, used it to generate random words one character at a time, and checked how many of the generated words are real.

> __Key Takeaways:__
>
> * __The data set the transition probabilities:__ We estimated each row of the transition matrix from the first two characters of the words that start with that character. The largest probabilities, such as `u` followed by `n`, come from how many English words begin.
>
> * __Each character depends only on the one before it:__ We generated words by drawing each character from the row of the transition matrix for the character before it. The words look like English in places, but the generator remembers nothing else about the word so far.
>
> * __Longer generated words are rarely real:__ We checked the generated words against the vocabulary and found that only a small fraction of them are real. The shortest words were real most often, and the fraction fell quickly as the words got longer.

We can now change the starting character, the word length, or the seed and see how the generated words respond.

___